# Сентимент-анализ
Модель: `mxlcw/rubert-tiny2-russian-financial-sentiment`
Score = p(positive) - p(negative)

In [ ]:
!pip install transformers torch -q


In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import re
import csv

FILE_PATH    = '/content/articles_labeled_long.csv'
MODEL_NAME = 'blanchefort/rubert-base-cased-sentiment-rusentiment'
OUTPUT_PATH = '/content/sentiment_blanchefort.csv'
CHECKPOINT  = '/content/sentiment_blanchefort_checkpoint.csv'
BATCH_SIZE   = 32
MAX_TOKENS   = 512
MAX_CHUNKS   = 4

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU" if device == 0 else "CPU"}')


Device: GPU


In [ ]:
# Загрузка данных

df = pd.read_csv(FILE_PATH, sep=';', on_bad_lines='skip',
                 encoding='utf-8-sig', lineterminator='\n', engine='c')
df = df[df['text'].notna() & (df['text'].str.len() > 100)].copy()
df = df.reset_index(drop=True)
print(f'Загружено: {len(df)} строк')
print(f'Колонки: {df.columns.tolist()}')


Загружено: 13136 строк
Колонки: ['ticker', 'date', 'title', 'domain', 'text', 'num']


In [ ]:
# Очистка текста — убираем шаблонные блоки
TRIM_AT = [
    'покупка ценных бумаг',
    'rbc group',
    'поделиться скопировать ссылку',
    'читайте рбк инвестиции в telegram',
    'покупка ценных бумаг',
    'вложение денежных средств',
    'получить полный доступ',
    'курсы валют публикуем',
    'самые дешевые наличные',
    'rbc group',
    'поделиться скопировать ссылку',
    'читайте рбк инвестиции в telegram',
    'официальные курсы доллара',
    'используются расчетов контрактам',
    'учитывались вклады сумму',
    'условий новых денег',
    'ставки указаны эффективном',
]

def trim_boilerplate(text):
    text_lower = str(text).lower()
    for trigger in TRIM_AT:
        idx = text_lower.find(trigger)
        if idx != -1:
            text = text[:idx]
            text_lower = text_lower[:idx]
    return text.strip()

print('Чистим тексты...')
df['text_clean'] = df['text'].apply(trim_boilerplate)
print('Готово')


Чистим тексты...
Готово


In [ ]:
# Загрузка модели
print(f'Загружаем модель {MODEL_NAME}...')
classifier = pipeline(
    'text-classification',
    model=MODEL_NAME,
    device=device,
    top_k=None,       # возвращаем все три класса
    truncation=True,
    max_length=MAX_TOKENS,
)
print('Модель загружена')


Загружаем модель blanchefort/rubert-base-cased-sentiment-rusentiment...


config.json:   0%|          | 0.00/952 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: blanchefort/rubert-base-cased-sentiment-rusentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/495 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Модель загружена


In [ ]:
# Токенизатор для разбивки на чанки
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def get_chunks(text, max_tokens=MAX_TOKENS, max_chunks=MAX_CHUNKS):
    """Разбивает текст на чанки по max_tokens токенов, берём первые max_chunks."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, min(len(tokens), max_tokens * max_chunks), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
    return chunks if chunks else [text[:500]]

def score_from_output(model_output):
    probs = {item['label'].upper(): item['score'] for item in model_output}
    return probs.get('POSITIVE', 0) - probs.get('NEGATIVE', 0)

def sentiment_score(text):
    """Считаем score как среднее по чанкам."""
    chunks = get_chunks(text)
    scores = []
    for chunk in chunks:
        try:
            out = classifier(chunk, truncation=True, max_length=512)[0]
            scores.append(score_from_output(out))
        except Exception:
            continue
    return sum(scores) / len(scores) if scores else None

print('Функции готовы')


Функции готовы


In [ ]:
# Основной цикл с чекпоинтом
import os

# Загружаем чекпоинт если есть
if os.path.exists(CHECKPOINT):
    done = pd.read_csv(CHECKPOINT, encoding='utf-8-sig')
    done_idx = set(done['original_idx'].tolist())
    results = done.to_dict('records')
    print(f'Продолжаем с чекпоинта: {len(done_idx)} уже обработано')
else:
    done_idx = set()
    results = []
    print('Начинаем с нуля')

todo = df[~df.index.isin(done_idx)]
print(f'Осталось обработать: {len(todo)} строк')

SAVE_EVERY = 400

for i, (idx, row) in enumerate(todo.iterrows()):
    score = sentiment_score(row['text_clean'])
    results.append({
        'original_idx': idx,
        'ticker':       row.get('ticker'),
        'date':         row.get('date'),
        'title':        row.get('title'),
        'domain':       row.get('domain'),
        'sentiment_score': score,
    })

    if (i + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False, encoding='utf-8-sig')
        print(f'  {i+1}/{len(todo)} обработано, чекпоинт сохранён')

# Финальное сохранение
df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'\nГотово! Сохранено: {OUTPUT_PATH}')
print(f'Строк: {len(df_out)}')
print(f'\nСтатистика sentiment_score:')
print(df_out['sentiment_score'].describe())


Начинаем с нуля
Осталось обработать: 13136 строк
  400/13136 обработано, чекпоинт сохранён
  800/13136 обработано, чекпоинт сохранён
  1200/13136 обработано, чекпоинт сохранён
  1600/13136 обработано, чекпоинт сохранён
  2000/13136 обработано, чекпоинт сохранён
  2400/13136 обработано, чекпоинт сохранён
  2800/13136 обработано, чекпоинт сохранён
  3200/13136 обработано, чекпоинт сохранён
  3600/13136 обработано, чекпоинт сохранён
  4000/13136 обработано, чекпоинт сохранён
  4400/13136 обработано, чекпоинт сохранён
  4800/13136 обработано, чекпоинт сохранён
  5200/13136 обработано, чекпоинт сохранён
  5600/13136 обработано, чекпоинт сохранён
  6000/13136 обработано, чекпоинт сохранён
  6400/13136 обработано, чекпоинт сохранён
  6800/13136 обработано, чекпоинт сохранён
  7200/13136 обработано, чекпоинт сохранён
  7600/13136 обработано, чекпоинт сохранён
  8000/13136 обработано, чекпоинт сохранён
  8400/13136 обработано, чекпоинт сохранён
  8800/13136 обработано, чекпоинт сохранён
  9200/

In [ ]:
# Агрегируем по дням

df_out['date'] = df['date'].values

daily = df_out.groupby(['ticker', 'date']).agg(
    sentiment_mean = ('sentiment_score', 'mean'),
    sentiment_std  = ('sentiment_score', 'std'),
    n_articles     = ('sentiment_score', 'count'),
).reset_index()

daily.to_csv('/content/sentiment_daily_blanchefort.csv', index=False, encoding='utf-8-sig', sep=';')
print(f'Дневная агрегация сохранена: {len(daily)} строк')
print(daily.head(10))

Дневная агрегация сохранена: 5123 строк
  ticker                 date  sentiment_mean  sentiment_std  n_articles
0   AFLT  2025-01-05 00:00:00   -1.054688e-02            NaN           1
1   AFLT  2025-01-12 00:00:00   -2.528621e-03            NaN           1
2   AFLT  2025-01-13 00:00:00    3.294554e-07            NaN           1
3   AFLT  2025-01-14 00:00:00   -2.787475e-03       0.002304           3
4   AFLT  2025-01-15 00:00:00   -5.994374e-03            NaN           1
5   AFLT  2025-01-19 00:00:00             NaN            NaN           0
6   AFLT  2025-01-20 00:00:00    1.688127e-03       0.001485           3
7   AFLT  2025-01-21 00:00:00   -2.966105e-03            NaN           1
8   AFLT  2025-01-22 00:00:00    8.334926e-04       0.002034           2
9   AFLT  2025-01-23 00:00:00   -3.504470e-03            NaN           1
